# UC1 — Multi-condition causal signaling with AnnNet as the hub

An end-to-end biology pipeline where **one AnnNet object is the central data
structure**: it holds the prior-knowledge network, the per-patient multi-omics
state, the causal inferences drawn from them, and the hand-off to downstream ML —
without ever leaving a single graph.

The story, notebook by notebook:

1. **setup** — load the OmniPath signed-directed prior-knowledge network (PKN).
2. **omics** — turn CPTAC LUAD transcriptomics and phosphoproteomics into
   tumour-normal contrasts, then infer per-patient TF and kinase activities.
3. **layers** — add one Kivelä patient layer per tumour; select cohorts.
4. **carnival** — solve a sparse signed-causal subnetwork per patient (CORNETO).
5. **consensus** — aggregate the cohort into a consensus layer and wire the
   inter-layer coupling that makes it genuinely multilayer.
6. **validation** — degree-bias, null-cohort, and pathway sanity checks.
7. **interop** — igraph PageRank, PyG/CX2 export, provenance diff, and a GNN.

Run `01`–`08` in order. Each notebook reloads `data/uc1.annnet` via
`uc1_common.load()`; heavy intermediate state (contrasts, activities, CARNIVAL
solutions) passes through the artifact files that `uc1_common` names. The series
is a software case study — it reports sanity checks, not biological discovery.

In [1]:
import annnet as an
from uc1_common import *  # noqa: F403

an.info()

Version,v0.2.0
License,BSD-3-Clause
Authors,"Youssef Zerta ✉, Daniele Bottazzi ✉, Denes Turei ✉"
Repository,https://github.com/saezlab/annnet
Documentation,https://saezlab.github.io/annnet/reference/
Default adapter backend,networkx
Default plot backend,graphviz
Graph backends,✓networkx✓igraph✓graph-tool✓pyg
Plot backends,✓graphviz✗pydot✓matplotlib
Tabular data backends,✓polars✓pandas✓pyarrow
I/O modules,✓annnet✓json/ndjson✓dataframes✓csv✓excel✓graphml/gexf✓sif✓cx2✓parquet✓zarr✓sbml✓scverse✓omnipath


## Load the OmniPath prior-knowledge network

`an.from_omnipath` fetches the signed, directional PKN together with the
annotation sources the later stages read (cancer-gene flags, kinase roles,
pathways). We use it as a reusable prior, not as tumour-specific ground truth.

In [ ]:
G = an.from_omnipath(
    dataset='omnipath',
    query={'organism': 'human', 'genesymbols': True},
    source_col='source_genesymbol',
    target_col='target_genesymbol',
    edge_attr_cols=[
        'is_stimulation', 'is_inhibition', 'consensus_direction',
        'consensus_stimulation', 'consensus_inhibition',
        'n_sources', 'n_references', 'sources',
    ],
    slice='omnipath_pkn',
    vertex_annotation_sources=[
        'HGNC', 'CancerGeneCensus', 'SignaLink_function', 'SignaLink_pathway',
        'UniProt_location', 'IntOGen', 'kinase.com', 'Phosphatome',
    ],
    load_vertex_annotations=True,
)
G.history.enable(True)
G.history.snapshot('after_pkn_load')
print(f'PKN loaded: |V|={G.global_count("vertices"):,} |E|={G.global_count("edges"):,}')
print(f'Active slice: {G.slices.active} | edge attrs: {list(G.edge_attributes.columns)}')

## PKN diagnostics

Before any patient-specific inference, separate strongly supported prior edges
(≥3 sources with a consensus direction) from the rest, and record each vertex's
prior degree — the sanity checks in `06` compare consensus frequency against it.

In [ ]:
import pandas as pd

edges = pkn_edges(G)
trusted = edges[
    (edges['n_sources'].fillna(0) >= 3) & (edges['consensus_direction'].fillna(False).astype(bool))
]
degree = pkn_degree(edges)

pkn_summary = pd.DataFrame({
    'metric': ['total_edges', 'trusted_edges', 'trusted_fraction', 'unique_vertices_with_degree'],
    'value': [len(edges), len(trusted), len(trusted) / max(len(edges), 1), degree.index.nunique()],
})
pkn_summary.to_csv(TABLES / 'pkn_summary.csv', index=False)
degree.rename_axis('vertex_id').reset_index().to_csv(TABLES / 'pkn_degree.csv', index=False)

G.write(str(SNAPSHOT), overwrite=True)
print(f'saved {SNAPSHOT}')
pkn_summary